In [9]:
import torch
import torch.nn as nn
from torch.nn import functional as F
import random


In [103]:
config = {
    "data_path": "hindi_english.txt",
    "d_model": 64, "num_heads": 4, "d_ff": 128, "num_layers": 2,
    "dropout": 0.1, "max_len": 100,
    "lr": 5e-4, "batch_size": 64, "num_epochs": 10, "seed": 42,
}

In [3]:
def load_pairs_from_txt(file_path):
    pairs = []
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                hindi, english = line.rstrip("\n").split("\t")
                pairs.append((hindi, english))
    return pairs

In [12]:
pairs_data = load_pairs_from_txt("hindi_english.txt")
print(len(pairs_data))
print(pairs_data[0])

10000
('आदमी रात को दफ़्तर में फूल बेचता है ।', 'The man sells flowers in the office at night.')


In [16]:
def split_data_test_train(pairs, train_frac=0.8, val_frac=0.1, seed=42):
    pairs_copy = pairs.copy()
    pairs_data = pairs[:]
    random.Random(seed).shuffle(pairs_data)

    pairs_len = len(pairs_data)
    train_size = int(pairs_len * train_frac)
    val_size = int(pairs_len * val_frac)

    train_pairs = pairs_data[:train_size]
    validation_pairs = pairs_data[train_size:train_size + val_size]
    test_pairs = pairs_data[train_size + val_size:]

    return train_pairs, validation_pairs, test_pairs

In [18]:
train_pairs, val_pairs, test_pairs = split_data_test_train(pairs_data)
print("Train pairs: " , len(train_pairs))
print("Validation pairs: ", len(val_pairs))
print("Test pairs:", len(test_pairs))

Train pairs:  8000
Validation pairs:  1000
Test pairs: 1000


In [100]:
import re

def tokenize_text_line(sentence):
    sentence = re.sub(r"\s*।\s*", " । ", sentence)    # exactly one space on each side of the danda
    return sentence.split()

In [25]:
def build_vocab_from_sentences(sentences):
    special_tokens = ['<PAD>', '<SOS>', '<EOS>', '<UNK>']
    all_tokens = sorted({ s_tok for sentence in sentences for s_tok in tokenize_text_line(sentence) })
    itos = special_tokens + all_tokens
    stoi = {tok : i_tok for i_tok , tok in enumerate(itos)}
    return stoi, itos

In [23]:
def encode_sentence(sentence, stoi):
    tokens = tokenize_text_line(sentence)
    encoded = [stoi.get(token, stoi['<UNK>']) for token in tokens]
    encoded = [stoi['<SOS>']] + encoded + [stoi['<EOS>']]
    return encoded

In [24]:
def decode_sequence(encoded_ids, itos):
    words = [ itos[i] for i in encoded_ids if i<len(itos) and itos[i] not in ['<PAD>', '<SOS>', '<EOS>']]
    return ' '.join(words)

In [26]:
# test the functions for encoding and decoding
src_stoi, src_itos = build_vocab_from_sentences([p[0] for p in train_pairs])
tgt_stoi, tgt_itos = build_vocab_from_sentences([p[1] for p in train_pairs])

print(len(src_stoi), len(tgt_stoi))

ids = encode_sentence(train_pairs[0][1], tgt_stoi)
print(ids)
print(decode_sequence(ids, tgt_itos))   # should print the original English sentence

77 98
[1, 9, 78, 57, 14, 46, 34, 59, 2]
They sell milk at home every morning.


In [34]:
def pad_sequences(encoded_ids,pad_token_id):
    max_length = max(len(ids) for ids in encoded_ids)
    padded = []
    for ids in encoded_ids:
        needed_padding = max_length - len(ids)
        padded_ids = ids + [pad_token_id] * needed_padding
        padded.append(padded_ids)
    tensor = torch.tensor(padded, dtype=torch.long)
    return tensor

In [35]:
def make_batch(pairs, src_stoi, tgt_stoi):
    src_encoded = [encode_sentence(pair[0], src_stoi) for pair in pairs]
    tgt_encoded = [encode_sentence(pair[1], tgt_stoi) for pair in pairs]

    tgt_in_encoded  = [ids[:-1] for ids in tgt_encoded]   # <SOS> w1 w2 ...   (drops <EOS>)
    tgt_out_encoded = [ids[1:]  for ids in tgt_encoded]   # w1 w2 ... <EOS>   (drops <SOS>)

    src_padded     = pad_sequences(src_encoded,     src_stoi['<PAD>'])
    tgt_in_padded  = pad_sequences(tgt_in_encoded,  tgt_stoi['<PAD>'])
    tgt_out_padded = pad_sequences(tgt_out_encoded, tgt_stoi['<PAD>'])

    return src_padded, tgt_in_padded, tgt_out_padded

In [36]:
src, tgt_in, tgt_out = make_batch(train_pairs[:4], src_stoi, tgt_stoi)
print(src.shape, tgt_in.shape, tgt_out.shape)
print(decode_sequence(tgt_in[0].tolist(),  tgt_itos))
print(decode_sequence(tgt_out[0].tolist(), tgt_itos))
print(tgt_in[0]); print(tgt_out[0])

torch.Size([4, 11]) torch.Size([4, 11]) torch.Size([4, 11])
They sell milk at home every morning.
They sell milk at home every morning.
tensor([ 1,  9, 78, 57, 14, 46, 34, 59,  0,  0,  0])
tensor([ 9, 78, 57, 14, 46, 34, 59,  2,  0,  0,  0])


In [38]:
def get_batches(pairs, batch_size, src_stoi, tgt_stoi, shuffle=True):
    pairs = pairs[:]
    if shuffle:
        random.shuffle(pairs)
    for i in range(0, len(pairs), batch_size):
        batch_pairs = pairs[i:i + batch_size]
        yield make_batch(batch_pairs, src_stoi, tgt_stoi)

In [39]:
for src, tgt_in, tgt_out in get_batches(train_pairs, 64, src_stoi, tgt_stoi):
    print(src.shape, tgt_in.shape, tgt_out.shape)
    break
# expect a first dimension of 64

torch.Size([64, 12]) torch.Size([64, 12]) torch.Size([64, 12])


In [44]:
def make_pad_mask(encoded_ids, pad_token_id):
    # True = real token (may be attended to), False = padding
    mask = (encoded_ids != pad_token_id)
    return mask.unsqueeze(1).unsqueeze(2)       # [B, 1, 1, L]

In [41]:
def make_causal_mask(encoded_ids):
    # Create a causal mask for the target sequence
    seq_len = encoded_ids.size(1)
    causal_mask = torch.tril(torch.ones((seq_len, seq_len), dtype=torch.bool))
    return causal_mask.unsqueeze(0).unsqueeze(1)  # Add batch and head dimensions

In [42]:
def make_tgt_mask(encoded_ids, pad_token_id):
    pad_mask = make_pad_mask(encoded_ids, pad_token_id)
    causal_mask = make_causal_mask(encoded_ids)
    combined_mask = pad_mask & causal_mask
    return combined_mask

In [45]:
src_mask = make_pad_mask(src, src_stoi['<PAD>'])
print(src_mask.shape, src_mask.dtype)       # [64, 1, 1, 12]  torch.bool

print(make_causal_mask(tgt_in[:, :4])[0, 0].int())   # the 4x4 lower triangle

tgt_mask = make_tgt_mask(tgt_in, tgt_stoi['<PAD>'])
print(tgt_mask.shape)                       # [64, 1, 12, 12]
print(tgt_mask[0, 0].int())                 # columns of padded positions are all 0

torch.Size([64, 1, 1, 12]) torch.bool
tensor([[1, 0, 0, 0],
        [1, 1, 0, 0],
        [1, 1, 1, 0],
        [1, 1, 1, 1]], dtype=torch.int32)
torch.Size([64, 1, 12, 12])
tensor([[1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0],
        [1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0],
        [1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0]], dtype=torch.int32)


In [ ]:
dim_model  = 64      # size of every token's vector
num_of_heads  = 4       # attention heads (d_model must divide evenly: 64 / 4 = 16)
dim_ff     = 128     # hidden size inside the feed-forward layer
num_of_layers = 2       # number of encoder layers and of decoder layers
dropout  = 0.1
max_len  = 100     # longest sequence the positional encoding supports

In [53]:
import math
import torch.nn as nn

class TokenEmbedding(nn.Module):
    def __init__(self, vocab_size, d_model, pad_id):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model, padding_idx=pad_id)
        self.dim_model = d_model

    def forward(self, x):
        # x: [B, L]  (batch of sequences of token IDs)
        return self.embedding(x) * math.sqrt(self.dim_model)  # scale by sqrt(d_model)


In [55]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len, dropout):
        super().__init__()
        self.dropout = nn.Dropout(dropout)

        pe = torch.zeros(max_len, d_model)
        position = torch.arange(max_len).unsqueeze(1)                       # [max_len, 1]
        div_term = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model))
        # your job: fill even columns with sin(position * div_term), odd columns with cos(...)
        pe[:, 0::2] = torch.sin(position * div_term)  # even indices
        pe[:, 1::2] = torch.cos(position * div_term)  # odd
        pe = pe.unsqueeze(0)                                                # [1, max_len, d_model]
        self.register_buffer("pe", pe)      # saved with the model, but NOT learned

    def forward(self, x):                   # x: [B, L, d_model]
        # your job: add the first L rows of pe to x, then apply dropout
        x = x + self.pe[:, :x.size(1)]     # [B, L, d_model]
        return self.dropout(x)              # [B, L, d_model]

In [56]:
emb = TokenEmbedding(len(src_stoi), dim_model, src_stoi['<PAD>'])
x = emb(src)
print(x.shape)                      # [64, 12, 64]

pos_enc = PositionalEncoding(dim_model, max_len, dropout)
y = pos_enc(x)
print(y.shape)                      # [64, 12, 64]  (same shape, positions added)

print(pos_enc.pe[0, 0, :4])         # tensor([0., 1., 0., 1.])

torch.Size([64, 12, 64])
torch.Size([64, 12, 64])
tensor([0., 1., 0., 1.])


In [57]:
def scaled_dot_product_attention(query, key, value, mask=None):
    d_k = query.size(-1)
    scores = torch.matmul(query, key.transpose(-2, -1)) / math.sqrt(d_k)  # [B, H, L_q, L_k]

    if mask is not None:
        scores = scores.masked_fill(mask == 0, -1e9)  # Mask out padding positions

    attention_weights = F.softmax(scores, dim=-1)  # [B, H, L_q, L_k]
    output = torch.matmul(attention_weights, value)  # [B, H, L_q, d_v]
    return output, attention_weights

In [58]:
q = torch.randn(2, 4, 5, 16)          # [B=2, heads=4, L=5, d_k=16]
k = torch.randn(2, 4, 5, 16)
v = torch.randn(2, 4, 5, 16)

out, w = scaled_dot_product_attention(q, k, v)
print(out.shape, w.shape)             # [2,4,5,16]  [2,4,5,5]
print(w.sum(dim=-1)[0, 0])            # all ones: each row of weights sums to 1

mask = make_causal_mask(torch.zeros(2, 5))      # [1,1,5,5] lower triangle
out, w = scaled_dot_product_attention(q, k, v, mask)
print(w[0, 0])                        # zeros above the diagonal; the first row is [1, 0, 0, 0, 0]

torch.Size([2, 4, 5, 16]) torch.Size([2, 4, 5, 5])
tensor([1.0000, 1.0000, 1.0000, 1.0000, 1.0000])
tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.4164, 0.5836, 0.0000, 0.0000, 0.0000],
        [0.3678, 0.2159, 0.4163, 0.0000, 0.0000],
        [0.1653, 0.1648, 0.3009, 0.3691, 0.0000],
        [0.1407, 0.0449, 0.7666, 0.0230, 0.0247]])


In [63]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)

    def split_heads(self, x):               # [B, L, d_model] -> [B, H, L, d_k]
        B, L, _ = x.size()
        return x.view(B, L, self.num_heads, self.d_k).transpose(1, 2)

    def forward(self, query, key, value, mask=None):
        B, Lq, _ = query.size()
        q = self.split_heads(self.W_q(query))
        k = self.split_heads(self.W_k(key))
        v = self.split_heads(self.W_v(value))
        out, weights = scaled_dot_product_attention(q, k, v, mask)
        out = out.transpose(1, 2).contiguous().view(B, Lq, self.num_heads * self.d_k)  # steps 5 and 6: join the heads, then W_o
        out = self.W_o(out)
        return out

In [64]:
mha = MultiHeadAttention(dim_model, num_of_heads)

x = torch.randn(2, 5, 64)
print(mha(x, x, x).shape)                      # [2, 5, 64]   self-attention

dec = torch.randn(2, 3, 64)                    # decoder side: 3 words
enc = torch.randn(2, 7, 64)                    # encoder side: 7 words
print(mha(dec, enc, enc).shape)                # [2, 3, 64]   cross-attention: output follows the QUERY's length

print(sum(p.numel() for p in mha.parameters()))   # 16640

torch.Size([2, 5, 64])
torch.Size([2, 3, 64])
16640


In [69]:
class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout):
        super().__init__()
        self.linear1 = nn.Linear(d_model, d_ff)
        self.linear2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.linear2(self.dropout(F.relu(self.linear1(x))))
        ## linear1 -> ReLU -> dropout -> linear2


class EncoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads)
        self.ff = FeedForward(d_model, d_ff, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, src_mask):
        x = self.norm1(x + self.dropout(self.self_attn(x, x, x, src_mask)))
        x = self.norm2(x + self.dropout(self.ff(x)))
        return x

In [70]:
ff = FeedForward(dim_model, dim_ff, dropout)
print(sum(p.numel() for p in ff.parameters()))        # 16576

layer = EncoderLayer(dim_model, num_of_heads, dim_ff, dropout)
print(sum(p.numel() for p in layer.parameters()))     # 33472  (attention 16640 + ff 16576 + 2 norms of 128)

# integration test on a real batch, using everything built so far
x = pos_enc(emb(src))                                 # [64, 12, 64]
src_mask = make_pad_mask(src, src_stoi['<PAD>'])
out = layer(x, src_mask)
print(out.shape)                                      # [64, 12, 64]  shape unchanged

print(out.mean(dim=-1)[0, :3])                        # all close to 0
print(out.std(dim=-1, unbiased=False)[0, :3])         # all close to 1

16576
33472
torch.Size([64, 12, 64])
tensor([ 1.4901e-08,  2.6077e-08, -1.0245e-08], grad_fn=<SliceBackward0>)
tensor([1.0000, 1.0000, 1.0000], grad_fn=<SliceBackward0>)


In [71]:
class Encoder(nn.Module):
    def __init__(self, vocab_size, d_model, num_heads, d_ff, num_layers, dropout, max_len, pad_id):
        super().__init__()
        self.embedding = TokenEmbedding(vocab_size, d_model, pad_id)
        self.pos_enc = PositionalEncoding(d_model, max_len, dropout)
        self.layers = nn.ModuleList([
            EncoderLayer(d_model, num_heads, d_ff, dropout) for _ in range(num_layers)
        ])

    def forward(self, src, src_mask):
        x = self.pos_enc(self.embedding(src))  # embedding, then positional encoding
        for layer in self.layers:
            x = layer(x, src_mask)
        return x

In [72]:
class DecoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads)
        self.cross_attn = MultiHeadAttention(d_model, num_heads)
        self.ff = FeedForward(d_model, d_ff, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, enc_out, src_mask, tgt_mask):
        x = self.norm1(x + self.dropout(self.self_attn(x, x, x, tgt_mask)))
        x = self.norm2(x + self.dropout(self.cross_attn(x, enc_out, enc_out, src_mask)))
        x = self.norm3(x + self.dropout(self.ff(x)))
        return x

In [73]:
enc = Encoder(len(src_stoi), dim_model, num_of_heads, dim_ff, num_of_layers,
              dropout, max_len, src_stoi['<PAD>'])
src_mask = make_pad_mask(src, src_stoi['<PAD>'])
enc_out = enc(src, src_mask)
print(enc_out.shape)                                   # [64, 12, 64]
print(sum(p.numel() for p in enc.parameters()))        # 71872 = 77*64 + 2*33472

tgt_emb = TokenEmbedding(len(tgt_stoi), dim_model, tgt_stoi['<PAD>'])
tgt_pos = PositionalEncoding(dim_model, max_len, dropout)
tgt_mask = make_tgt_mask(tgt_in, tgt_stoi['<PAD>'])

dec_layer = DecoderLayer(dim_model, num_of_heads, dim_ff, dropout)
x = tgt_pos(tgt_emb(tgt_in))
print(dec_layer(x, enc_out, src_mask, tgt_mask).shape) # [64, 12, 64]
print(sum(p.numel() for p in dec_layer.parameters()))  # 50240 = 2*16640 + 16576 + 3*128

torch.Size([64, 12, 64])
71872
torch.Size([64, 12, 64])
50240


In [76]:
class Decoder(nn.Module):
    def __init__(self, vocab_size, d_model, num_heads, d_ff, num_layers, dropout, max_len, pad_id):
        super().__init__()
        self.embedding = TokenEmbedding(vocab_size, d_model, pad_id)
        self.pos_enc = PositionalEncoding(d_model, max_len, dropout)
        self.layers = nn.ModuleList([
            DecoderLayer(d_model, num_heads, d_ff, dropout) for _ in range(num_layers)
        ])

    def forward(self, tgt_in, enc_out, src_mask, tgt_mask):
        x = self.pos_enc(self.embedding(tgt_in))
        for layer in self.layers:
            x = layer(x, enc_out, src_mask, tgt_mask)
        return x

In [77]:
class Transformer(nn.Module):
    def __init__(self, src_vocab, tgt_vocab, d_model, num_heads, d_ff, num_layers,
                 dropout, max_len, src_pad_id, tgt_pad_id):
        super().__init__()
        self.src_pad_id = src_pad_id
        self.tgt_pad_id = tgt_pad_id
        self.encoder = Encoder(src_vocab, d_model, num_heads, d_ff, num_layers, dropout, max_len, src_pad_id)
        self.decoder = Decoder(tgt_vocab, d_model, num_heads, d_ff, num_layers, dropout, max_len, tgt_pad_id)
        self.output_layer = nn.Linear(d_model, tgt_vocab)

    def forward(self, src, tgt_in):
        src_mask = make_pad_mask(src, self.src_pad_id)
        tgt_mask = make_tgt_mask(tgt_in, self.tgt_pad_id)
        enc_out = self.encoder(src, src_mask)
        dec_out = self.decoder(tgt_in, enc_out, src_mask, tgt_mask)
        return self.output_layer(dec_out)  # logits: [B, T, tgt_vocab]

In [78]:
model = Transformer(len(src_stoi), len(tgt_stoi), dim_model, num_of_heads, dim_ff,
                    num_of_layers, dropout, max_len, src_stoi['<PAD>'], tgt_stoi['<PAD>'])

logits = model(src, tgt_in)
print(logits.shape)                                   # [64, 12, 98]
print(sum(p.numel() for p in model.parameters()))     # 184994

probs = torch.softmax(logits, dim=-1)
print(probs.sum(dim=-1)[0, :3])                       # all ones
print(probs[0, 0].max().item())                       # small, around 0.01 to 0.03

torch.Size([64, 12, 98])
184994
tensor([1.0000, 1.0000, 1.0000], grad_fn=<SliceBackward0>)
0.03135282173752785


In [79]:
model.eval()                                  # switch dropout off so results are repeatable
with torch.no_grad():
    l1 = model(src, tgt_in)
    tgt2 = tgt_in.clone()
    tgt2[:, 5] = 7                            # change the word at position 5
    l2 = model(src, tgt2)

print(torch.allclose(l1[:, :5], l2[:, :5], atol=1e-5))    # True: positions 0-4 can't see position 5
print(torch.allclose(l1[:, 5:], l2[:, 5:], atol=1e-5))    # False: positions 5+ do change

True
False


In [80]:
criterion = nn.CrossEntropyLoss(ignore_index=tgt_stoi['<PAD>'])

def compute_loss(logits, tgt_out, criterion):
    V = logits.size(-1)
    return criterion(logits.reshape(-1, V), tgt_out.reshape(-1))

In [81]:
optimizer = torch.optim.Adam(model.parameters(), lr=5e-4, betas=(0.9, 0.98), eps=1e-9)

In [82]:
def training_step(model, batch, criterion, optimizer):
    src, tgt_in, tgt_out = batch
    model.train()                   # dropout ON
    logits = model(src, tgt_in)                    # 1. forward
    loss = compute_loss(logits, tgt_out, criterion)                      # 2. loss
    optimizer.zero_grad()           # 3.
    loss.backward()                            # 4. backward
    nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)                             # clip gradients
    optimizer.step()                             # 5. optimizer step
    return loss.item()              # a plain Python number for logging

In [84]:
batch = make_batch(train_pairs[:64], src_stoi, tgt_stoi)
print(training_step(model, batch, criterion, optimizer))     # about 4.6

4.744685173034668


In [86]:
for step in range(150):
    loss = training_step(model, batch, criterion, optimizer)
    if step % 10 == 0:
        print(step, round(loss, 4))

0 4.6531
10 3.8402
20 3.3729
30 2.9377
40 2.5545
50 2.2167
60 1.926
70 1.6998
80 1.497
90 1.2897
100 1.1302
110 0.9706
120 0.8382
130 0.7313
140 0.6174


In [112]:
def train_one_epoch(model, pairs, batch_size, src_stoi, tgt_stoi, criterion, optimizer):
    losses = []
    for batch in get_batches(pairs, batch_size, src_stoi, tgt_stoi, shuffle=True):
        loss = training_step(model, batch, criterion, optimizer)
        losses.append(loss)
    return sum(losses) / len(losses)


def evaluate(model, pairs, batch_size, src_stoi, tgt_stoi, criterion):
    model.eval()
    losses = []
    with torch.no_grad():
        for src, tgt_in, tgt_out in get_batches(pairs, batch_size, src_stoi, tgt_stoi, shuffle=False):
            logits = model(src, tgt_in)
            loss = compute_loss(logits, tgt_out, criterion)
            losses.append(loss.item())
    return sum(losses) / len(losses)


""" def train_model(train_pairs, val_pairs, src_stoi, tgt_stoi, num_epochs, batch_size, lr):
    model = Transformer(len(src_stoi), len(tgt_stoi), dim_model, num_of_heads, dim_ff,
                    num_of_layers, dropout, max_len, src_stoi['<PAD>'], tgt_stoi['<PAD>'])               # fresh model every call
    criterion = nn.CrossEntropyLoss(ignore_index=tgt_stoi['<PAD>'])
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, betas=(0.9, 0.98), eps=1e-9)

    best_val = float("inf")
    for epoch in range(1, num_epochs + 1):
        train_loss = train_one_epoch(model, train_pairs, batch_size, src_stoi, tgt_stoi, criterion, optimizer)
        val_loss = evaluate(model, val_pairs, batch_size, src_stoi, tgt_stoi, criterion)
        print(f"epoch {epoch:2d} | train {train_loss:.4f} | val {val_loss:.4f}")
        if val_loss < best_val:
            best_val = val_loss
            torch.save(model.state_dict(), "best_model.pt")
    return model """

def train_model(train_pairs, val_pairs, src_vocab, tgt_vocab, config):
    src_stoi, src_itos = src_vocab
    tgt_stoi, tgt_itos = tgt_vocab

    model = build_model(config, src_stoi, tgt_stoi)
    criterion = nn.CrossEntropyLoss(ignore_index=tgt_stoi['<PAD>'])
    optimizer = torch.optim.Adam(model.parameters(), lr=config["lr"], betas=(0.9, 0.98), eps=1e-9)

    best_val = float("inf")
    for epoch in range(1, config["num_epochs"] + 1):
        train_loss = train_one_epoch(model, train_pairs, config["batch_size"],
                                     src_stoi, tgt_stoi, criterion, optimizer)
        val_loss = evaluate(model, val_pairs, config["batch_size"],
                            src_stoi, tgt_stoi, criterion)
        print(f"epoch {epoch:2d} | train {train_loss:.4f} | val {val_loss:.4f}")
        if val_loss < best_val:
            best_val = val_loss
            save_checkpoint("best_model.pt", model, src_itos, tgt_itos, config)
    return model    

In [94]:
model = train_model(train_pairs, val_pairs, src_stoi, tgt_stoi, num_epochs=10, batch_size=64, lr=5e-4)

epoch  1 | train 2.4188 | val 1.0300
epoch  2 | train 0.6740 | val 0.2676
epoch  3 | train 0.1928 | val 0.0630
epoch  4 | train 0.0629 | val 0.0303
epoch  5 | train 0.0294 | val 0.0128
epoch  6 | train 0.0136 | val 0.0066
epoch  7 | train 0.0076 | val 0.0043
epoch  8 | train 0.0052 | val 0.0024
epoch  9 | train 0.0033 | val 0.0013
epoch 10 | train 0.0024 | val 0.0008


In [95]:
def translate(model, sentence, src_stoi, tgt_stoi, tgt_itos, max_len=20):
    model.eval()
    with torch.no_grad():
        src = torch.tensor([encode_sentence(sentence, src_stoi)])        # [1, S]
        src_mask = make_pad_mask(src, src_stoi['<PAD>'])
        enc_out = model.encoder(src, src_mask)                           # encoder, run once

        tgt = torch.tensor([[tgt_stoi['<SOS>']]])                        # [1, 1]
        for _ in range(max_len):
            tgt_mask = make_tgt_mask(tgt, tgt_stoi['<PAD>'])
            dec_out = model.decoder(tgt, enc_out, src_mask, tgt_mask)
            next_logits = model.output_layer(dec_out[:, -1])             # [1, vocab]
            next_id = torch.argmax(next_logits, dim=-1)                    # argmax
            tgt = torch.cat([tgt, next_id.view(1, 1)], dim=1)
            if next_id.item() == tgt_stoi['<EOS>']:
                break
    return decode_sequence(tgt[0].tolist(), tgt_itos)

In [96]:
best = Transformer(len(src_stoi), len(tgt_stoi), dim_model, num_of_heads, dim_ff,
                   num_of_layers, dropout, max_len, src_stoi['<PAD>'], tgt_stoi['<PAD>'])
best.load_state_dict(torch.load("best_model.pt"))

<All keys matched successfully>

In [ ]:
for hi, en in test_pairs[:5]:
    print("Hindi :", hi)
    print("Ref from the file :", en)
    print("Translated output from the model :", translate(best, hi, src_stoi, tgt_stoi, tgt_itos))
    print()

print(translate(best, "केशव स्कूल जाता है ।", src_stoi, tgt_stoi, tgt_itos))

Hindi : औरत रोज़ दफ़्तर में पानी पीती है ।
Ref from the file : The woman drinks water in the office every day.
Translated output from the model : The woman drinks water in the office every day.

Hindi : शिक्षक रोज़ घर पर रोटी खाता है ।
Ref from the file : The teacher eats bread at home every day.
Translated output from the model : The teacher eats bread at home every day.

Hindi : औरत सुबह स्कूल में पानी पीती है ।
Ref from the file : The woman drinks water at school in the morning.
Translated output from the model : The woman drinks water at school in the morning.

Hindi : वे हर सुबह चावल खाते हैं ।
Ref from the file : They eat rice every morning.
Translated output from the model : They eat rice every morning.

Hindi : शिक्षक हर सुबह घर पर किताब खरीदता है ।
Ref from the file : The teacher buys a book at home every morning.
Translated output from the model : The teacher buys a book at home every morning.

The boy writes a story. at school.


In [101]:
def exact_match_accuracy(model, pairs, src_stoi, tgt_stoi, tgt_itos):
    correct = 0
    mistakes = []
    for hi, en in pairs:
        out = translate(model, hi, src_stoi, tgt_stoi, tgt_itos)
        ref = " ".join(tokenize_text_line(en))      # same tokenizing as the model output
        if out == ref:                                     # your job: compare out and ref
            correct += 1
        else:
            mistakes.append((hi, ref, out))
    return correct / len(pairs), mistakes

In [102]:
criterion = nn.CrossEntropyLoss(ignore_index=tgt_stoi['<PAD>'])
print("test loss:", evaluate(best, test_pairs, 64, src_stoi, tgt_stoi, criterion))
acc, mistakes = exact_match_accuracy(best, test_pairs, src_stoi, tgt_stoi, tgt_itos)
print("exact match:", acc)
for m in mistakes[:5]: print(m)       # look at the failures, if any

test loss: 0.0007691347564104944
exact match: 1.0


In [106]:
def build_model(config, src_stoi, tgt_stoi):
    return Transformer(len(src_stoi), len(tgt_stoi), config["d_model"], config["num_heads"],
                       config["d_ff"], config["num_layers"], config["dropout"], config["max_len"],
                       src_stoi['<PAD>'], tgt_stoi['<PAD>'])

In [107]:
def save_checkpoint(path, model, src_itos, tgt_itos, config):
    torch.save({
        "model_state": model.state_dict(),
        "src_itos": src_itos,
        "tgt_itos": tgt_itos,
        "config": config,
    }, path)

def load_checkpoint(path):
    ckpt = torch.load(path)
    src_itos, tgt_itos, config = ckpt["src_itos"], ckpt["tgt_itos"], ckpt["config"]
    src_stoi = {token: i for i, token in enumerate(src_itos)}
    tgt_stoi = {token: i for i, token in enumerate(tgt_itos)}
    model = build_model(config, src_stoi, tgt_stoi)
    model.load_state_dict(ckpt["model_state"])
    model.eval()
    return model, src_stoi, tgt_stoi, tgt_itos

In [ ]:
def main():
    random.seed(config["seed"]); torch.manual_seed(config["seed"])

    pairs = load_pairs_from_txt(config["data_path"])
    train_pairs, val_pairs, test_pairs = split_data_test_train(pairs)

    src_vocab = build_vocab_from_sentences([p[0] for p in train_pairs])    # train only
    tgt_vocab = build_vocab_from_sentences([p[1] for p in train_pairs])

    train_model(train_pairs, val_pairs, src_vocab, tgt_vocab, config)       # saves best checkpoint

    model, src_stoi, tgt_stoi, tgt_itos = load_checkpoint("best_model.pt")
    # test loss + exact match here (step 1)

    while True:                                                              # interactive queries
        q = input("Hindi> ").strip()
        if q in ("", "quit"):
            break
        print(translate(model, q, src_stoi, tgt_stoi, tgt_itos))

if __name__ == "__main__":
    main()

epoch  1 | train 2.4144 | val 1.1082
epoch  2 | train 0.7209 | val 0.2881
epoch  3 | train 0.2140 | val 0.0685
epoch  4 | train 0.0618 | val 0.0256
epoch  5 | train 0.0243 | val 0.0104
epoch  6 | train 0.0119 | val 0.0061
epoch  7 | train 0.0072 | val 0.0036
epoch  8 | train 0.0046 | val 0.0020
epoch  9 | train 0.0026 | val 0.0010
epoch 10 | train 0.0024 | val 0.0008
